### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\chinm\RAG_PROJECT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def process_all_pdfs():
    """Process all PDF files in the RAG project pdf_directory"""

    all_documents = []
    pdf_dir = Path(r"C:\Users\chinm\RAG_PROJECT\pdf_directory")

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            # Load the PDF using PyMuPDFLoader
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Split the documents into smaller chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200
            )

            split_docs = text_splitter.split_documents(documents)

            all_documents.extend(split_docs)

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    return all_documents


# Run the function
documents = process_all_pdfs()
print(f"Total chunks created: {len(documents)}")

Found 2 PDF files to process
Processing C:\Users\chinm\RAG_PROJECT\pdf_directory\Attention is all you need_research paper.pdf
Processing C:\Users\chinm\RAG_PROJECT\pdf_directory\IEEE_Research_paper_organized.pdf
Total chunks created: 97


### Embedding & Vectorstore DB

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings 
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name (str): HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of documents
        
        Args:
            documents (List[str]): List of strings to embed
            
        Returns:
            numpy array of embeddings with shape(len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings generated successfully")
        return embeddings

    def get_embedding_dim(self) -> int:
        """
        Get the dimension of the embeddings generated by the mode
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot get embedding dimension.")
        return self.model.get_sentence_embedding_dimension()
    
 ## initialize the embedding manager
embedding_manager = EmbeddingManager()  
embedding_manager 

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4783.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


### Vector Store

In [5]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "chroma_db"):
        """
        Initialize the vector store

        Args:
            collection_name (str): Name of the collection in ChromaDB
            persist_directory (str): Directory to persist vector database
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            # New ChromaDB persistent client (correct modern syntax)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"}
            )

            print("ChromaDB client and collection initialized successfully")

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents (List[Any]): List of document objects
            embeddings (np.ndarray): Corresponding embeddings
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_texts.append(doc.page_content)

            # Convert numpy embedding to list
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_texts,
                metadatas=metadatas,
                embeddings=embeddings_list
            )

            print("Documents added to vector store successfully")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise e


# Initialize vector store
vectorstore = VectorStore()

print(vectorstore)

ChromaDB client and collection initialized successfully


In [6]:
# Convert documents to text and generate embeddings
documents_texts = [doc.page_content for doc in documents]
embeddings = embedding_manager.generate_embeddings(documents_texts)

# Add documents and embeddings to vector store
vectorstore.add_documents(documents, embeddings)

Generating embeddings for 97 texts...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches: 100%|██████████| 4/4 [00:04<00:00,  1.21s/it]


Embeddings generated successfully
Adding 97 documents to the vector store...
Documents added to vector store successfully


### Retriever Pipeline From VectorStore

In [7]:
from typing import List, Dict, Any

class RAGRetriever:
    """Retrieves relevant documents from the vector store based on a query"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"]   # FIXED
            )

            retrieved_docs = []

            if results and results.get("documents") and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]  # ids come automatically

                for doc_id, document, metadata, distance in zip(ids, documents, metadatas, distances):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score
                        })

                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold")

            else:
                print("No documents retrieved for the query.")

            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []
        
# Initialize RAG retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [8]:
rag_retriever

In [9]:
rag_retriever.retrieve("What is attention all you need")

Retrieving documents for query: 'What is attention all you need'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.88it/s]

Embeddings generated successfully
Retrieved 2 documents above the score threshold


[{'id': 'doc_3e8fd265_12',
  'document': '3.2\nAttention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'title': '',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'subject': '',
   'file_path': 'C:\\Users\\chinm\\RAG_PROJECT\\pdf_directory\\Attention is all you need_research paper.pdf',
   'keywords': '',
   'producer': 'pdfTeX-1.40.25',
   'author': '',
   'doc_index': 12,
   'creator': 'LaTeX with hyperref',
   'trapped': '',
   'moddate': '2024-04-10T21:11:43+00:00',
   'total_pages': 15,
   'creationDate': 'D:20240410211143Z',
   'page': 2,
   'content_length': 216,
   'modDate': 'D:20240410211143Z',
   'source': 'C:\\Users\\chinm\\RAG_PROJECT\\pdf_directory\\Attention is all you need_research paper.pdf',
   'format': 'PDF 1.5'},
  'similarity_score': 0.20985376834869385},
 {'id': 'doc_7b4547fa_49',
  'd

### Integration Vectordb Context pipeline With LLM output

In [10]:
### Simple RAG pipeline with Gemini LLM

import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize Gemini LLM
gemini_api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=gemini_api_key)

llm = genai.GenerativeModel("gemini-2.5-flash")


# Simple RAG function : retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([result["document"] for result in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question"

    # Generate answer using Gemini
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm.generate_content(prompt)

    return response.text

C:\Users\chinm\AppData\Local\Temp\ipykernel_9372\3518593057.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [11]:
answer = rag_simple("What is attention function?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is attention function?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.31it/s]

Embeddings generated successfully
Retrieved 3 documents above the score threshold


An attention function maps a query and a set of key-value pairs to an output. The query, keys, values, and output are all vectors, and the output is computed as a weighted sum.


### Enhanced RAG

In [12]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize Gemini
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

llm = genai.GenerativeModel("gemini-2.5-flash")


def enhanced_rag(query, retriever, llm, top_k=5, debug=False):
    """
    Enhanced RAG pipeline:
    Retrieval → Context Ranking → Prompt Engineering → LLM Response
    """

    # STEP 1: Retrieve documents
    results = retriever.retrieve(query, top_k=top_k)

    if not results:
        return "No relevant documents found."

    # STEP 2: Sort by similarity score (best first)
    results = sorted(results, key=lambda x: x["similarity_score"], reverse=True)

    # STEP 3: Build context
    context_chunks = []
    sources = []

    for r in results:
        context_chunks.append(r["document"])
        sources.append(r["metadata"].get("source", "unknown"))

    context = "\n\n".join(context_chunks)

    if debug:
        print("\nRetrieved Context:")
        for r in results:
            print(f"Score: {r['similarity_score']:.3f}")

    # STEP 4: Better prompt
    prompt = f"""
You are an AI assistant answering questions using retrieved documents.

Rules:
- Only answer using the provided context.
- If the answer is not in the context, say "The answer is not found in the provided documents."
- Keep answers concise and clear.

Context:
{context}

Question:
{query}

Answer:
"""

    # STEP 5: Generate response
    response = llm.generate_content(prompt)

    answer = response.text.strip()

    return {
        "answer": answer,
        "sources": list(set(sources))
    }

In [13]:
answer = enhanced_rag("What is random forest?",rag_retriever,llm)

print("Answer:\n", answer["answer"])
print("\nSources:", answer["sources"])

Retrieving documents for query: 'What is random forest?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.42it/s]


Embeddings generated successfully
Retrieved 1 documents above the score threshold
Answer:
 Random Forest primarily stabilises predictions and reduces the variance that arises from XGBoost’s aggressive learning. It uses 300 trees, maximum depth: none, and has bootstrap sampling enabled.

Sources: ['C:\\Users\\chinm\\RAG_PROJECT\\pdf_directory\\IEEE_Research_paper_organized.pdf']
